# B2B MRO Supply Chain Analytics
## Procurement, fulfilment, and customer engagement

**Author:** Marziyeh Eslamparasti — Business Analyst, Hamburg  
**Tools:** Python · Pandas · DuckDB (SQL) · Scikit-learn  
**Data:** Reproducible synthetic scenario for a fictional German MRO distributor

I built this notebook around five linked tables: customers, products, suppliers, customer orders, and supplier orders.

> This is a synthetic scenario. I use it to demonstrate the analysis workflow, not as evidence about real companies, customer groups, cities, or countries.

## 1. Setup and data checks

Run `python data_simulation.py` from the repository root before executing this notebook. The setup below locates the project root automatically and reads the generated files from `data/`.

In [1]:
from pathlib import Path
import sys
import warnings

import duckdb
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)


def find_project_root(start: Path) -> Path:
    # Locate the repository root from the current working directory.
    for candidate in (start, *start.parents):
        if (candidate / "data_simulation.py").exists() and (candidate / "notebooks").exists():
            return candidate
    raise FileNotFoundError("Could not locate the project root.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
DATA_DIR = PROJECT_ROOT / "data"
FIGURE_DIR = PROJECT_ROOT / "reports" / "figures"
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from utils import compute_rfm, summarise_delivery
from visualizations import (
    save_clustering,
    save_customer_engagement,
    save_delivery_performance,
    save_kpi_dashboard,
    save_product_analysis,
    save_rfm_analysis,
    save_supplier_performance,
)

FILES = {
    "customer_orders": "customer_orders.csv",
    "customers": "customers.csv",
    "products": "products.csv",
    "suppliers": "suppliers.csv",
    "supplier_orders": "supplier_orders.csv",
}
missing_files = [name for name in FILES.values() if not (DATA_DIR / name).exists()]
if missing_files:
    missing = "\n- ".join(missing_files)
    raise FileNotFoundError(
        f"Missing generated files in {DATA_DIR}:\n- {missing}\n"
        "Run: python data_simulation.py"
    )

customer_orders = pd.read_csv(
    DATA_DIR / FILES["customer_orders"], parse_dates=["order_date"]
)
customers = pd.read_csv(DATA_DIR / FILES["customers"])
mro_products = pd.read_csv(DATA_DIR / FILES["products"])
supplier_master = pd.read_csv(DATA_DIR / FILES["suppliers"])
supplier_orders = pd.read_csv(
    DATA_DIR / FILES["supplier_orders"], parse_dates=["order_date"]
)

print(f"Customer orders: {len(customer_orders):,}")
print(f"Supplier orders: {len(supplier_orders):,}")
print(f"Customers:       {len(customers):,}")
print(f"Products:        {len(mro_products):,}")
print(f"Suppliers:       {len(supplier_master):,}")

Customer orders: 20,000
Supplier orders: 15,000
Customers:       500
Products:        300
Suppliers:       30


In [2]:
expected_rows = {
    "customer_orders": 20_000,
    "supplier_orders": 15_000,
    "customers": 500,
    "products": 300,
    "suppliers": 30,
}
actual_rows = {
    "customer_orders": len(customer_orders),
    "supplier_orders": len(supplier_orders),
    "customers": len(customers),
    "products": len(mro_products),
    "suppliers": len(supplier_master),
}
assert actual_rows == expected_rows, f"Unexpected table sizes: {actual_rows}"

return_mismatch = (
    customer_orders["is_returned"].eq(1)
    & customer_orders["return_reason"].eq("Not returned")
) | (
    customer_orders["is_returned"].eq(0)
    & customer_orders["return_reason"].ne("Not returned")
)
assert not return_mismatch.any(), "Return flags and reasons are inconsistent."
assert customer_orders["customer_id"].isin(customers["customer_id"]).all()
assert supplier_orders["supplier_id"].isin(supplier_master["supplier_id"]).all()
assert supplier_orders["product_id"].isin(mro_products["product_id"]).all()

print("Data-integrity checks passed.")

Data-integrity checks passed.


## 2. KPI overview

In [3]:
kpis = pd.DataFrame(
    {
        "Metric": [
            "Customer orders",
            "Supplier orders",
            "Simulated revenue",
            "Simulated gross profit",
            "Average basket",
            "Customer-order on-time rate",
            "Supplier-order on-time rate",
            "Active customers",
        ],
        "Result": [
            f"{len(customer_orders):,}",
            f"{len(supplier_orders):,}",
            f"€{customer_orders['order_value'].sum():,.0f}",
            f"€{customer_orders['gross_profit'].sum():,.0f}",
            f"{customer_orders['basket_size'].mean():.2f} items",
            f"{customer_orders['on_time'].mean():.1%}",
            f"{supplier_orders['on_time'].mean():.1%}",
            f"{customer_orders['customer_id'].nunique():,}",
        ],
    }
)
print(kpis.to_string(index=False))
save_kpi_dashboard(
    customer_orders,
    supplier_orders,
    FIGURE_DIR / "chart1_kpi_dashboard.png",
)

                     Metric      Result
            Customer orders      20,000
            Supplier orders      15,000
          Simulated revenue €35,210,924
     Simulated gross profit €11,267,495
             Average basket  3.32 items
Customer-order on-time rate       56.1%
Supplier-order on-time rate       88.3%
           Active customers         500


![Scenario KPI dashboard](../reports/figures/chart1_kpi_dashboard.png)

Revenue and gross profit come from the simulation and are included to demonstrate the KPI structure.

## 3. Customer engagement and basket size

In [4]:
basket_by_level = (
    customer_orders.groupby("technician_level")
    .agg(
        avg_basket=("basket_size", "mean"),
        avg_order_value=("order_value", "mean"),
        orders=("order_id", "count"),
        return_rate=("is_returned", "mean"),
    )
    .sort_values("avg_order_value", ascending=False)
)
basket_by_level["return_rate"] *= 100

basket_by_work = (
    customer_orders.groupby("work_situation")
    .agg(
        avg_basket=("basket_size", "mean"),
        avg_order_value=("order_value", "mean"),
        orders=("order_id", "count"),
    )
    .sort_values("avg_order_value", ascending=False)
)

engineer_basket = basket_by_level.loc["Maintenance Engineer", "avg_basket"]
junior_basket = basket_by_level.loc["Junior Technician", "avg_basket"]
print(f"Maintenance Engineer / Junior basket ratio: {engineer_basket / junior_basket:.2f}x\n")
print("By technician level")
print(basket_by_level.round(2).to_string())
print("\nBy work situation")
print(basket_by_work.round(2).to_string())
save_customer_engagement(
    customer_orders,
    FIGURE_DIR / "chart2_customer_engagement.png",
)

Maintenance Engineer / Junior basket ratio: 2.80x

By technician level
                      avg_basket  avg_order_value  orders  return_rate
technician_level                                                      
Maintenance Engineer        5.37          4129.34    4598         5.18
Senior Technician           3.24          1391.35    9218         5.54
Junior Technician           1.92           549.61    6184         5.38

By work situation
                  avg_basket  avg_order_value  orders
work_situation                                       
Facility Manager        4.60          2553.99    6875
Company Employee        2.89          1526.87    7483
Freelance               2.34          1103.63    5642


![Customer engagement](../reports/figures/chart2_customer_engagement.png)

Larger expected baskets for experienced technicians and facility managers are built into the generator. This run measures the resulting difference at about 2.8× between Maintenance Engineers and Junior Technicians. Return rates are similar across all three levels.

With real data, I would test bundles or compatibility guidance and measure their effect on basket size, margin, and returns.

## 4. Customer-order delivery performance

In [5]:
city_delivery = summarise_delivery(customer_orders, "city").sort_values("on_time_rate")
late_delays = customer_orders.loc[customer_orders["on_time"].eq(0), "delay_days"]

print(city_delivery.to_string(index=False))
print()
print(f"City on-time spread: {city_delivery['on_time_rate'].max() - city_delivery['on_time_rate'].min():.1f} percentage points")
print(f"Mean delay among late orders: {late_delays.mean():.1f} days")
print(f"Median delay among late orders: {late_delays.median():.1f} days")
print(f"Maximum delay: {late_delays.max():.0f} days")
print(f"Late orders delayed more than 7 days: {(late_delays > 7).sum():,}")
save_delivery_performance(
    customer_orders,
    FIGURE_DIR / "chart3_delivery_performance.png",
)

     city  order_count  on_time_rate  avg_delay_days
  Cologne         2144          53.7            0.81
  Dresden         1307          55.4            0.83
Nuremberg          953          55.4            0.80
Frankfurt         2156          56.3            0.79
  Hamburg         4189          56.3            0.77
   Munich         3602          56.4            0.75
   Berlin         3470          57.0            0.78
Stuttgart         2179          57.0            0.76

City on-time spread: 3.3 percentage points
Mean delay among late orders: 1.8 days
Median delay among late orders: 1.0 days
Maximum delay: 10 days
Late orders delayed more than 7 days: 30


![Customer-order delivery performance](../reports/figures/chart3_delivery_performance.png)

City rates range from 53.7% to 57.0%. Because every city uses the same delay distribution, this narrow spread is sampling variation. The low overall rate comes from the scenario setting rather than an observed logistics network.

## 5. SQL analysis

I use DuckDB to run the business queries directly over the Pandas tables.

In [6]:
db = duckdb.connect()
db.register("orders_tbl", customer_orders)
db.register("supplier_orders_tbl", supplier_orders)
db.register("suppliers_tbl", supplier_master)

basket_growth = db.execute(
    """
    WITH city_year AS (
        SELECT
            city,
            order_year,
            AVG(basket_size) AS avg_basket
        FROM orders_tbl
        GROUP BY city, order_year
    )
    SELECT
        city,
        ROUND(MAX(CASE WHEN order_year = 2022 THEN avg_basket END), 2) AS avg_basket_2022,
        ROUND(MAX(CASE WHEN order_year = 2023 THEN avg_basket END), 2) AS avg_basket_2023,
        ROUND(
            (MAX(CASE WHEN order_year = 2023 THEN avg_basket END)
             / MAX(CASE WHEN order_year = 2022 THEN avg_basket END) - 1) * 100,
            1
        ) AS change_pct
    FROM city_year
    GROUP BY city
    ORDER BY change_pct DESC
    """
).df()

print("Average basket change by city")
print(basket_growth.to_string(index=False))

Average basket change by city
     city  avg_basket_2022  avg_basket_2023  change_pct
  Cologne             2.92             3.01         3.0
   Berlin             3.26             3.26         0.0
   Munich             3.29             3.28        -0.1
Nuremberg             3.02             3.01        -0.4
  Hamburg             3.58             3.52        -1.5
Stuttgart             3.38             3.31        -2.0
  Dresden             3.79             3.71        -2.0
Frankfurt             3.34             3.17        -4.9


In [7]:
supplier_exposure = db.execute(
    """
    SELECT
        so.supplier_id,
        s.supplier_country,
        ROUND(s.reliability_score * 100, 1) AS assumed_service_score_pct,
        COUNT(so.sup_order_id) AS order_count,
        ROUND(SUM(so.order_value), 0) AS procurement_value,
        ROUND(AVG(so.actual_lead_days), 1) AS avg_actual_lead_days,
        ROUND(AVG(so.on_time) * 100, 1) AS simulated_on_time_pct
    FROM supplier_orders_tbl AS so
    JOIN suppliers_tbl AS s USING (supplier_id)
    WHERE s.reliability_score < 0.80
    GROUP BY so.supplier_id, s.supplier_country, s.reliability_score
    HAVING SUM(so.order_value) > 50000
    ORDER BY procurement_value DESC
    LIMIT 10
    """
).df()

print("Priority-review supplier relationships in the scenario")
print(supplier_exposure.to_string(index=False))
save_supplier_performance(
    supplier_orders,
    supplier_master,
    FIGURE_DIR / "chart4_supplier_performance.png",
)

Priority-review supplier relationships in the scenario
supplier_id supplier_country  assumed_service_score_pct  order_count  procurement_value  avg_actual_lead_days  simulated_on_time_pct
    SUP-021            China                       77.0          487         11285790.0                  21.4                   51.3
    SUP-004            China                       76.0          485         11148474.0                  22.9                   49.7
    SUP-025      Netherlands                       71.0          514         11106229.0                   2.2                   99.6
    SUP-014            China                       68.0          501         10915884.0                  27.2                   36.7
    SUP-022            China                       67.0          455         10574972.0                  22.0                   44.0
    SUP-030            China                       62.0          469          9912347.0                  26.9                   34.8


![Supplier performance](../reports/figures/chart4_supplier_performance.png)

“Priority review” combines high procurement exposure with a low assumed service score. It is a rule for this scenario, not a real supplier-risk rating.

In [8]:
snapshot_date = pd.Timestamp("2024-01-01")
recency_accounts = db.execute(
    """
    WITH account_summary AS (
        SELECT
            customer_id,
            MAX(order_date) AS last_order_date,
            COUNT(order_id) AS lifetime_orders,
            SUM(order_value) AS lifetime_value
        FROM orders_tbl
        GROUP BY customer_id
    )
    SELECT
        customer_id,
        last_order_date,
        lifetime_orders,
        ROUND(lifetime_value, 0) AS lifetime_value,
        DATE_DIFF('day', last_order_date, DATE '2024-01-01') AS days_since_order
    FROM account_summary
    WHERE DATE_DIFF('day', last_order_date, DATE '2024-01-01') > 90
    ORDER BY lifetime_value DESC
    """
).df()

print(f"Accounts with more than 90 days since last simulated order: {len(recency_accounts)}")
print(recency_accounts.head(10).to_string(index=False))

Accounts with more than 90 days since last simulated order: 7
customer_id last_order_date  lifetime_orders  lifetime_value  days_since_order
  CUST-0120      2023-09-22               51        141815.0               101
  CUST-0472      2023-09-20               29        131473.0               103
  CUST-0330      2023-09-06               30         84410.0               117
  CUST-0137      2023-08-17               34         26516.0               137
  CUST-0394      2023-09-28               19         23308.0                95
  CUST-0317      2023-09-29               41         23150.0                94
  CUST-0044      2023-09-26               33         15376.0                97


In [9]:
segment_matrix = db.execute(
    """
    SELECT
        technician_level,
        city,
        COUNT(order_id) AS orders,
        ROUND(AVG(basket_size), 2) AS avg_basket,
        ROUND(AVG(order_value), 0) AS avg_order_value,
        ROUND(AVG(on_time) * 100, 1) AS on_time_pct,
        ROUND(AVG(is_returned) * 100, 1) AS return_pct
    FROM orders_tbl
    GROUP BY technician_level, city
    ORDER BY avg_order_value DESC
    LIMIT 15
    """
).df()

print(segment_matrix.to_string(index=False))

    technician_level      city  orders  avg_basket  avg_order_value  on_time_pct  return_pct
Maintenance Engineer   Dresden     384        6.24           4839.0         56.0         5.5
Maintenance Engineer    Munich     661        5.40           4316.0         58.1         4.4
Maintenance Engineer Frankfurt     531        5.91           4212.0         53.5         4.9
Maintenance Engineer   Hamburg    1229        5.28           4197.0         56.6         4.9
Maintenance Engineer Stuttgart     476        5.77           4179.0         55.9         5.9
Maintenance Engineer    Berlin     717        5.02           3972.0         59.8         5.4
Maintenance Engineer   Cologne     376        4.86           3826.0         52.7         7.2
Maintenance Engineer Nuremberg     224        4.21           2701.0         53.1         3.6
   Senior Technician    Berlin    1687        3.38           1490.0         56.2         5.7
   Senior Technician   Hamburg    1811        3.37           1458.0   

## 6. Product and procurement view

In [10]:
category_procurement = (
    supplier_orders.groupby("category")
    .agg(
        purchase_orders=("sup_order_id", "count"),
        procurement_value=("order_value", "sum"),
        avg_actual_lead_days=("actual_lead_days", "mean"),
        on_time_rate=("on_time", "mean"),
    )
    .sort_values("procurement_value", ascending=False)
)
category_procurement["on_time_rate"] *= 100
print(category_procurement.round(2).to_string())
save_product_analysis(
    supplier_orders,
    FIGURE_DIR / "chart5_product_analysis.png",
)

                       purchase_orders  procurement_value  avg_actual_lead_days  on_time_rate
category                                                                                     
Diagnostic Equipment              1520       1.066186e+08                  8.20         87.96
Hydraulic Parts                   1542       6.141400e+07                  8.41         88.39
Power Tools                       1477       4.000010e+07                  8.17         89.71
Pneumatic Tools                   1606       3.355770e+07                  8.33         88.36
Electrical Components             1287       2.918577e+07                  8.70         87.18
Safety Equipment                  1553       1.837992e+07                  8.62         87.89
Hand Tools                        1811       1.429855e+07                  8.52         88.57
Lubricants & Fluids               1388       9.738811e+06                  8.47         87.90
Cutting & Abrasives               1319       8.931928e+06   

![Product and category analysis](../reports/figures/chart5_product_analysis.png)

This category view shows how I would structure a procurement scorecard. The values depend on the generated product mix and supplier orders.

## 7. RFM customer segmentation

RFM summarizes recency, frequency, and monetary value. I use it here to prioritize account review, not to predict churn.

In [11]:
rfm_scores = compute_rfm(customer_orders)
segment_summary = (
    rfm_scores.groupby("segment")
    .agg(
        customers=("customer_id", "count"),
        avg_recency=("recency", "mean"),
        avg_frequency=("frequency", "mean"),
        total_value=("monetary", "sum"),
    )
    .sort_values("total_value", ascending=False)
)

at_risk_value = rfm_scores.loc[rfm_scores["segment"].eq("At Risk"), "monetary"].sum()
print(segment_summary.round(1).to_string())
print(f"\nRevenue exposure in the descriptive At Risk segment: €{at_risk_value:,.0f}")
save_rfm_analysis(
    rfm_scores,
    FIGURE_DIR / "chart6_rfm_analysis.png",
)

              customers  avg_recency  avg_frequency  total_value
segment                                                         
Loyal               108         11.3           43.9   10713356.6
Champion             55          4.9           46.1    8036000.8
Potential           110         16.7           40.9    7489924.9
New Customer        139         23.6           37.4    6680852.1
At Risk              66         33.7           35.7    1891945.9
Lost                 22         52.1           30.4     398843.2

Revenue exposure in the descriptive At Risk segment: €1,891,946


![RFM analysis](../reports/figures/chart6_rfm_analysis.png)

For a real CRM decision, I would add contribution margin, service cost, account context, and measured response to previous outreach.

## 8. Exploratory K-Means segmentation

In [12]:
cluster_input = rfm_scores[["recency", "frequency", "monetary"]].copy()
feature_scaler = StandardScaler()
scaled_features = feature_scaler.fit_transform(cluster_input)

silhouette_results = {}
candidate_models = {}
for cluster_count in range(2, 6):
    candidate = KMeans(n_clusters=cluster_count, random_state=42, n_init=10)
    labels = candidate.fit_predict(scaled_features)
    silhouette_results[cluster_count] = silhouette_score(scaled_features, labels)
    candidate_models[cluster_count] = candidate

best_k = max(silhouette_results, key=silhouette_results.get)
rfm_scores["cluster"] = candidate_models[best_k].labels_
cluster_profiles = (
    rfm_scores.groupby("cluster")
    .agg(
        customers=("customer_id", "count"),
        avg_recency=("recency", "mean"),
        avg_frequency=("frequency", "mean"),
        avg_monetary=("monetary", "mean"),
    )
    .sort_values("avg_monetary", ascending=False)
)

print("Silhouette scores")
for cluster_count, score in silhouette_results.items():
    print(f"  k={cluster_count}: {score:.3f}")
print(f"Selected solution: k={best_k}")
print("\nCluster profiles")
print(cluster_profiles.round(1).to_string())
save_clustering(
    rfm_scores,
    silhouette_results,
    FIGURE_DIR / "chart7_clustering.png",
)

Silhouette scores
  k=2: 0.284
  k=3: 0.291
  k=4: 0.335
  k=5: 0.277
Selected solution: k=4

Cluster profiles
         customers  avg_recency  avg_frequency  avg_monetary
cluster                                                     
0               60         16.6           41.7      218503.1
2              153         13.5           46.5       57789.1
3               68         58.1           37.5       47497.6
1              219         13.6           35.8       45795.3


![Exploratory K-Means segmentation](../reports/figures/chart7_clustering.png)

Among k=2 to k=5, k=4 has the highest silhouette score (about 0.335). The separation is modest, so I treat the clusters as an exploratory view rather than four fixed customer types.

## 9. What I would do next

1. Test bundles or guided product selection and measure the change in basket size, margin, and returns.
2. Add carrier, route, capacity, and service-level fields before drawing conclusions about delivery performance.
3. Review supplier relationships using both procurement exposure and service measures.
4. Combine RFM with margin, service cost, and account context before planning retention activity.
5. Check cluster stability over time before using the segments in a commercial workflow.

## What is missing

- Real operational data or cited evidence for the scenario assumptions.
- Order-line detail linking baskets to individual products.
- Carrier, route, capacity, and service-level variables.
- Time-based segment validation and measured intervention results.

---

[LinkedIn](https://linkedin.com/in/marziyeh-eslamparasti) · [GitHub portfolio](https://github.com/marziyeh-ba)